# 06 — Final test evaluation (run ONCE)

This notebook evaluates the **frozen, exported** model on the sealed test participants **P009 and P010**.

Rules enforced in code:
* the model file `checkpoints/selected/gaze_model.pt` and `reports/model_selection_decision.json` must already
  exist (the decision is recorded before any test data is read);
* you must type the exact confirmation phrase; the test split is unlocked by a one-time token;
* no parameter is updated, no hyper-parameter, epoch or threshold is chosen, nothing is fitted on test images
  (ImageNet constants are fixed; no statistics are estimated);
* a second evaluation is refused unless you give an override reason, which is recorded permanently in
  `reports/test_unsealed.json`.

Calibrated results (§7) are a **separate experiment** with its own protocol and are reported separately.
GPU cells: §4 (prediction). §3 builds the test cache on the CPU (≈10 min).

In [ ]:
# --- Workspace setup -----------------------------------------------------------
# WHAT: makes the `src` package importable and loads training_config.yaml.
# WHY : every notebook uses the same code and the same configuration.
# PATHS: resolved from this notebook's folder, never from the terminal's cwd.
import sys
from pathlib import Path

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
if not (WORKSPACE / "src" / "config.py").is_file():
    raise RuntimeError(f"Open this notebook from training_workspace/notebooks (cwd = {Path.cwd()})")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.config import load_config
cfg = load_config(WORKSPACE / "training_config.yaml")
cfg.paths.ensure_output_dirs()
W, H = cfg.screen_px
print("Workspace     :", cfg.paths.workspace)
print("Dataset root  :", cfg.paths.dataset_root, "(read-only)")
print("Screen        :", W, "x", H, "px")

In [ ]:
import json, numpy as np, pandas as pd, torch
from src.utils import read_json, write_json, select_device
from src.dataset import require_audit_pass
require_audit_pass(cfg)
MODEL_FILE = cfg.paths.checkpoints / "selected" / "gaze_model.pt"
DECISION = read_json(cfg.paths.reports / "model_selection_decision.json")
print("model:", MODEL_FILE, "exists:", MODEL_FILE.is_file())
print("decision recorded before testing:", DECISION)

## 1. Confirmation gate

In [ ]:
from src.evaluation import open_test_gate, CONFIRMATION_PHRASE
CONFIRMATION = ""                    # <- type exactly: I confirm the model and all settings are final
OVERRIDE_REASON = None               # only if re-evaluating; recorded permanently
TOKEN = open_test_gate(cfg, MODEL_FILE, CONFIRMATION, DECISION, OVERRIDE_REASON)
print("Test split unlocked for this session.")

## 2. Load the frozen model (no training, no gradient)

In [ ]:
from src.infer import load_exported
hw = cfg["hardware"]
dev = select_device(hw["device"], hw["allow_cpu_training"], hw["precision"])
model, blob = load_exported(MODEL_FILE, dev.device)
for p in model.parameters():
    p.requires_grad_(False)
assert blob["preprocessing"]["version"] == cfg["preprocessing"]["version"], "preprocessing changed since export"
print(blob["selection"])

## 3. Build the test cache (same preprocessing; CPU, resumable)

In [ ]:
from src.pipeline import build_cache_for_split
info = build_cache_for_split(cfg, "test", test_token=TOKEN)
print(info["normalized_ok"], "/", info["rows"], "normalised; failures:", info["failures"])

## 4. Predict every usable test frame — GPU

In [ ]:
from src.pipeline import predict_split
from src.evaluation import prediction_table
test_df, xy, gaze, track = predict_split(cfg, model, "test", dev.device, dev.precision, test_token=TOKEN)
test_pred = prediction_table(test_df, xy, W, H, str(MODEL_FILE), cfg["preprocessing"]["version"], DECISION["run"])
OUT = cfg.paths.predictions / "test"; OUT.mkdir(parents=True, exist_ok=True)
test_pred.to_parquet(OUT / "test_predictions.parquet", index=False)
test_pred.to_csv(OUT / "test_predictions.csv", index=False)
print(track)

## 5. Metrics — P009, P010 and pooled (uncalibrated)

In [ ]:
from src.evaluation import evaluate_predictions
test_eval = evaluate_predictions(test_pred, cfg, bootstrap=True)
test_eval["tracking"] = track
write_json(cfg.paths.reports / "test_evaluation.json", test_eval)
per = pd.DataFrame(test_eval["per_participant"]).set_index("participant_id")
pooled = pd.Series(test_eval["frame_level"], name="pooled")
pd.concat([per, pooled.to_frame().T])[["n", "mean_euclid_px", "median_euclid_px", "rmse_px", "p90_euclid_px",
    "p95_euclid_px", "max_euclid_px", "mean_abs_x_px", "mean_abs_y_px", "bias_x_px", "bias_y_px",
    "r2_x", "r2_y", "mean_euclid_pct_diag", "within_32px", "within_64px", "within_100px"]].round(3)

In [ ]:
print(json.dumps({"bootstrap_pooled": test_eval["bootstrap_mean_euclid"],
                   "bootstrap_per_participant": test_eval["per_participant_bootstrap"],
                   "trial_level": test_eval["trial_level"], "target_selection": test_eval["target_selection_success"],
                   "limitations": test_eval["limitations"]}, indent=1))

## 6. Test figures

In [ ]:
from IPython.display import Image, display
from src.visualization import (plot_error_analysis, plot_prediction_map, plot_error_heatmap,
                               plot_vector_field, interactive_prediction_map)
fd = cfg.paths.figures / "test"
figs = list(plot_error_analysis(test_pred, W, H, fd, "test").values())
figs += [plot_prediction_map(test_pred, W, H, fd / "test_map_all", "Test: P009 and P010"),
         plot_error_heatmap(test_pred, W, H, fd / "test_error_heatmap"),
         plot_vector_field(test_pred, W, H, fd / "test_vector_field")]
for pid, g in test_pred.groupby("participant_id"):
    figs.append(plot_prediction_map(g, W, H, fd / f"test_map_{pid}", f"Test {pid}", color_by="session_id"))
for f in figs: display(Image(filename=str(f)))
print(interactive_prediction_map(test_pred, W, H, OUT / "interactive_prediction_map_test.html", "Test predictions"))

## 7. Separate experiment: personalised calibration on test participants
Protocol fixed in advance (from the config, chosen on validation): calibration points from S01 only, scoring on
S02–S05 only. These numbers answer "how accurate is it **after** a k-point calibration?" and must be reported as
such, never mixed with §5.

In [ ]:
from src.calibration import evaluate_calibration
cal_test = evaluate_calibration(test_pred, cfg, cfg["calibration"]["point_counts"])
cal_test.to_csv(cfg.paths.reports / "test_calibration_separate_experiment.csv", index=False)
cal_test

## 8. Close the gate

In [ ]:
from src.evaluation import close_test_gate
close_test_gate(cfg, cfg.paths.reports / "test_evaluation.json")
print("Test evaluation recorded in", cfg.paths.reports / "test_unsealed.json")